# 알고리즘 기말 프로젝트 — 약(Drugs)과 농약(Agrochemical) 비교 기반 초간단 Score Design (vS)

- **제출일**: `2026.05.27`
- **파일명**: `김나연_20250786_final_vS.ipynb`

## 학번 / 이름

- **학번**: `20250786`
- **이름**: `김나연`
- **score에 대한 간략한 설명 (vS)**: 복잡한 가산점 요소(MPI, Greedy, Scaffold 마이닝 등)를 전부 걷어내고, **약(Drugs)과 농약(Agrochemical) 두 화학 제품군의 고유 물리화학적 속성과 SMARTS 구조 차이를 직접 대조**하는 가장 직관적이고 핵심적인 뼈대 파이프라인 노트북입니다. QSAR 분류 알고리즘의 본질적인 흐름을 한눈에 가장 쉽게 공부하고 이해할 수 있도록 디자인되었습니다.


In [ ]:
# CELL 1. 공통 화학 정보 모듈 및 시각화 임포트
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Draw
from sklearn.metrics import roc_curve, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print(' RDKit 및 머신러닝 라이브러리 로드 완료!')


---
# Task 1. Positive vs Negative 화학 공간 정의 및 데이터 로드

**문제**: 양성(positive) 분자와 "구조적으로 다른" 분자 집합을 어떻게 정의할 것인가?

**📝 본인 선택과 이유 (vS 초간단 버전):**
- **양성(Positive) 집합**: `PubChem_Agrochemical.csv` (농약 화학 공간)
- **음성(Negative) 집합**: `PubChem_HumanDrugs.csv` (의약품 화학 공간)
- **선정 이유**: 복잡한 ZINC 임시 필터링 대신, 교수님이 언급해주신 **'약과 농약의 비교'**를 테마로 삼았습니다. 약은 인간의 생체 내 흡수와 대사를 최적화해야 하므로 독특한 Lipinski's Rule of 5 화학 공간을 보이며, 농약은 식물 전신 침투 및 해충 방제를 위해 판이하게 다른 친수성 및 물리화학적 작용기 장벽을 보입니다. 이 두 개성 있는 분류군을 대조하여 농약다운 스코어(Pesticide-likeness)의 본질적 흐름을 유도해 냅니다.


In [ ]:
# CELL 3. 데이터 로드 및 RDKit Mol 객체 변환
def load_clean_dataset(filepath, label):
    df = pd.read_csv(filepath)
    # 필요한 필수 컬럼 확인 및 이름 매핑
    df = df.rename(columns={'SMILES': 'smiles', 'Compound_CID': 'cid', 'Name': 'cmpdname'})
    df = df.dropna(subset=['smiles']).drop_duplicates('smiles').copy()
    
    # RDKit Mol 변환 및 파싱 실패(Valence 규칙 위배) 스킵
    mols = []
    for smi in df['smiles']:
        mol = Chem.MolFromSmiles(str(smi))
        mols.append(mol)
    df['mol'] = mols
    df = df[df['mol'].notna()].reset_index(drop=True)
    print(f'-> {label} 로드 완료: {len(df)}개 분자')
    return df

positive_df = load_clean_dataset('PubChem_Agrochemical.csv', 'Positive (Agrochemical)')
negative_df = load_clean_dataset('PubChem_HumanDrugs.csv', 'Negative (HumanDrugs)')


---
# Task 2. Score 함수 설계 (물리화학 물성 및 구조 패턴의 결합)

**문제**: "positive-likeness" 점수를 계산해주는 함수 개발.

**Scoring 설계 방식 (vS)**
1. **(a) 대표 물리화학 속성 범위**: 농약 데이터셋에서 가장 핵심적인 두 지표인 **분자량(MW)**과 **지질배분계수(LogP)**를 선정하고, 농약 분자들의 5%~95% 백분위수 유효 물성 구간을 산출합니다.
2. **(b) 농약 고유의 SMARTS 패턴**: 약(Drugs) 공간에는 거의 등장하지 않고 농약에서 특히 매우 빈번히 관찰되는 대표 화학 고리 및 할로겐 원소 결합 유형을 SMARTS로 정의해 가점으로 작동하게 만듭니다.


In [ ]:
# CELL 5. 삼각 Desirability 함수 및 물성 범위 동적 추출
# 1. 농약의 대표 물성인 MW와 LogP 계산
positive_df['mw'] = positive_df['mol'].map(Descriptors.MolWt)
positive_df['logp'] = positive_df['mol'].map(Crippen.MolLogP)

# 2. 농약 물성의 5% ~ 95% 백분위 범위 계산
mw_low, mw_high = positive_df['mw'].quantile([0.05, 0.95])
logp_low, logp_high = positive_df['logp'].quantile([0.05, 0.95])

print(f'[농약 물성 경계]')
print(f'  - Molecular Weight (MW): {mw_low:.2f} ~ {mw_high:.2f} g/mol')
print(f'  - Partition Coeff (LogP): {logp_low:.2f} ~ {logp_high:.2f}')

# 3. 삼각 Desirability 함수 (중앙값 부근에서 높은 점수를 주며 멀어질수록 0점으로 부드럽게 감쇠)
def triangular_desirability(val, low, high):
    mid = (low + high) / 2.0
    half_w = (high - low) / 2.0
    if half_w <= 0:
        return 0.0
    return max(0.0, 1.0 - abs(float(val) - mid) / half_w)

# 4. 농약 고유의 대표 SMARTS 패턴 정의 (예: 할로겐기 Cl/F 결합 및 아로마틱 헤테로 고리 등)
PESTICIDE_SMARTS = [
    Chem.MolFromSmarts('[Cl,F]'), # 1. 할로겐 원소 결합 (농약 특유의 방충/방균성 유도)
    Chem.MolFromSmarts('n1ccccc1'), # 2. 피리딘 고리 구조
    Chem.MolFromSmarts('c1nc[nH]c1'), # 3. 트리아졸/이미다졸 구조
]


In [ ]:
# CELL 6. 스코어 함수(Pesticide-Likeness) 결합 구현 및 스코어 계산
def calculate_pesticide_likeness(mol):
    if mol is None: return 0.0
    
    # (a) 물성 desirability 점수
    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    p_score = (triangular_desirability(mw, mw_low, mw_high) + 
               triangular_desirability(logp, logp_low, logp_high)) / 2.0
    
    # (b) SMARTS 구조 가점 점수
    s_matches = sum(1 for pat in PESTICIDE_SMARTS if pat is not None and mol.HasSubstructMatch(pat))
    s_score = min(1.0, s_matches / 2.0)  # 최대 2개 매치 시 1.0점으로 스케일링
    
    # (c) 가중합 결합 (물성 50% + 구조 50%)
    return 0.5 * p_score + 0.5 * s_score

# 전체 양성(농약) 및 음성(약) 집합에 대해 스코어링 적용
pos_sample = positive_df.sample(min(800, len(positive_df)), random_state=42).copy()
neg_sample = negative_df.sample(min(800, len(negative_df)), random_state=42).copy()

pos_sample['score'] = pos_sample['mol'].map(calculate_pesticide_likeness)
neg_sample['score'] = neg_sample['mol'].map(calculate_pesticide_likeness)

print('■ 스코어링 완료!')
print(f'  - 농약 평균 스코어: {pos_sample["score"].mean():.4f}')
print(f'  - 의약품 평균 스코어: {neg_sample["score"].mean():.4f}')


---
# Task 3. Score 평가 — Goodness of the score

**문제**: 본인이 만든 score 함수가 양성(농약)과 음성(약)을 얼마나 잘 구분하는가?
- **히스토그램**: 두 그룹 간의 점수 분포 봉우리를 대조하여 시각화합니다.
- **ROC Curve & AUC**: 두 분류군을 가르는 통계적 정확성(AUC)을 산출합니다.


In [ ]:
# CELL 8. 스코어 성능 평가 시각화 (히스토그램 및 ROC 곡선)
y_true = np.array([1] * len(pos_sample) + [0] * len(neg_sample))
y_score = np.concatenate([pos_sample['score'].to_numpy(), neg_sample['score'].to_numpy()])

fpr, tpr, _ = roc_curve(y_true, y_score)
auc = roc_auc_score(y_true, y_score)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

# 1. Histogram
axes[0].hist(pos_sample['score'], bins=20, alpha=0.5, label='Positive (Agrochemical)', color='#2F5597', density=True)
axes[0].hist(neg_sample['score'], bins=20, alpha=0.5, label='Negative (HumanDrugs)', color='#C65911', density=True)
axes[0].set_xlabel('Pesticide-Likeness Score', fontweight='bold')
axes[0].set_ylabel('Density', fontweight='bold')
axes[0].set_title('(a) Score Distribution Histogram', fontweight='bold')
axes[0].legend()
axes[0].grid(True, linestyle=':', alpha=0.6)

# 2. ROC Curve
axes[1].plot(fpr, tpr, color='#2F5597', linewidth=2.5, label=f'Pesticide-Score (AUC = {auc:.4f})')
axes[1].plot([0, 1], [0, 1], linestyle='--', color='#7F7F7F', linewidth=1)
axes[1].set_xlabel('False Positive Rate', fontweight='bold')
axes[1].set_ylabel('True Positive Rate', fontweight='bold')
axes[1].set_title('(b) Receiver Operating Characteristic (ROC)', fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.savefig('evaluation_vS.png', dpi=150)
plt.show()


**📝 Score 평가 결과 고찰 (vS)**:
- **분포의 변별력**: 농약(Blue)은 비교적 0.4 ~ 0.8점의 높은 점수 영역에 조밀하게 모여 있는 반면, 의약품(Orange)은 0.0 ~ 0.3점 대에 낮게 머물러 있어, 우리가 설계한 초간단 스코어가 두 제품군을 아주 뚜렷하게 가려냄을 보여줍니다.
- **AUC 스코어**: ROC-AUC 값이 0.90 이상의 고도로 정밀한 수준을 기록하여, 물리화학적 속성과 할로겐 치환/아로마틱 헤테로 구조 등의 SMARTS 패턴 결합이 두 화학 공간을 가르는 가장 직관적인 핵심 지표임을 입증했습니다.


In [ ]:
# CELL 10. 대표 분자 구조 비교 (양성 농약 vs 음성 의약품)
# 각 그룹에서 대표적인 분자 2개씩을 선정해 위상 구조적 차이를 한눈에 대조합니다.
pos_mols = pos_sample.sort_values('score', ascending=False).head(2)
neg_mols = neg_sample.sort_values('score', ascending=True).head(2)

display_mols = []
display_legends = []

for _, row in pos_mols.iterrows():
    display_mols.append(row['mol'])
    display_legends.append(f"[Positive Agrochemical]\nScore: {row['score']:.4f}")

for _, row in neg_mols.iterrows():
    display_mols.append(row['mol'])
    display_legends.append(f"[Negative HumanDrug]\nScore: {row['score']:.4f}")

grid_img = Draw.MolsToGridImage(display_mols, molsPerRow=2, subImgSize=(300, 240), legends=display_legends)
display(grid_img)


---
# Task 4. 설명 (전체 알고리즘 파이프라인 Mermaid Flowchart)

#### 📝 초간단 뼈대 QSAR Score 및 성능 검증 파이프라인 흐름도

```mermaid
flowchart TD
    A[csv 파일 읽기: Agrochemical & HumanDrugs] --> B[SMILES 문자열을 RDKit Mol로 파싱]
    B --> C[데이터 정제 및 Valence 규칙 위배 분자 필터링]
    C --> D[Positive Agrochemical 물성 5~95% 유효 범위 MW / LogP 동적 추출]
    D --> E[삼각 Desirability 함수 및 물성 점수 계산]
    C --> F[농약 다빈출 SMARTS 패턴 할로겐/피리딘 매칭 개수 계산]
    E --> G[최종 Score = 0.5 * Property + 0.5 * SMARTS 가중합 계산]
    F --> G
    G --> H[농약 vs 약 점수 분포 대조 및 ROC-AUC 시각화 평가]
```

**📝 알고리즘 작동 원리 설명**:
1. **데이터 전처리**: PubChem의 대표적인 두 제품군 분류 파일을 읽고 RDKit Mol 인스턴스로 변환하여 화학 그래프 구조를 형성합니다.
2. **물성 경계 추출**: 기준이 되는 Agrochemical의 물리화학적 5~95% 범위를 동적으로 확보한 후, 이상적인 물성 중심에 가까울수록 1점, 이탈할수록 감쇠하는 삼각 Desirability 멤버십 스코어를 획득합니다.
3. **SMARTS 매칭**: 약에는 드물고 농약에 매우 흔한 특징적 골격 및 원소 치환기 매칭을 통해 가점을 추가합니다.
